### **Arquitectura reproducible de un agente multimodal**

#### **De una cadena fija a decisiones dinámicas con trazabilidad**

Este cuaderno compara tres diseños bajo las mismas tareas y herramientas.

1. Cadena fija,
2. Workflow con reglas,
3. Agente con selección dinámica de herramientas.

#### **Pregunta central**

¿Qué diferencia existe entre una cadena fija, un workflow y un agente que decide dinámicamente sus acciones?

#### **Hipótesis**

**H1.** La cadena fija tendrá mayor costo por llamadas innecesarias.

**H2.** El workflow será eficiente en tareas conocidas pero menos flexible ante combinaciones no previstas.

**H3.** El agente dinámico reducirá llamadas cuando el planificador sea correcto.

**H4.** Un límite explícito de pasos reducirá bucles y ejecuciones redundantes.

#### **Diseño experimental**

Los tres sistemas comparten tareas, herramientas, presupuesto y criterio de éxito.

Las modalidades se simulan con referencias estructuradas para mantener ejecución local y reproducible.

In [ ]:
from __future__ import annotations

import csv
import json
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable

In [ ]:
RESULTS_DIR = Path("results/cuaderno28_mcc225")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 228
random.seed(SEED)

print("Semilla fijada:", SEED)
print("Directorio de resultados:", RESULTS_DIR)

#### **Estado, observaciones y acciones**

El estado conserva presupuesto, pasos, observaciones, acciones y condición de terminación.

Esta separación permite analizar causalmente cada trayectoria.

In [ ]:
@dataclass
class Observation:
    modality: str
    content: str
    source: str
    confidence: float


@dataclass
class ActionRecord:
    step: int
    tool_name: str
    arguments: dict[str, Any]
    result: str
    cost: float
    success: bool


@dataclass
class AgentState:
    query: str
    observations: list[Observation] = field(default_factory=list)
    actions: list[ActionRecord] = field(default_factory=list)
    current_step: int = 0
    max_steps: int = 4
    remaining_budget: float = 4.0
    finished: bool = False
    final_answer: str | None = None
    status: str = "inicial"

In [ ]:
class TraceLogger:
    def __init__(self) -> None:
        self.records: list[dict[str, Any]] = []

    def add(self, event: str, payload: dict[str, Any]) -> None:
        record = {
            "evento": event,
            "marca_de_tiempo": time.time(),
            "datos": payload,
        }
        self.records.append(record)

    def save(self, path: Path) -> None:
        with path.open("w", encoding="utf-8") as file:
            for record in self.records:
                file.write(json.dumps(record, ensure_ascii=False) + "\n")

#### **Herramientas multimodales simuladas**

Cada herramienta declara nombre, modalidad, costo y comportamiento.

In [ ]:
@dataclass
class ToolResult:
    content: str
    cost: float
    success: bool


class Tool:
    def __init__(
        self,
        name: str,
        description: str,
        modality: str,
        cost: float,
        handler: Callable[[dict[str, Any]], str],
    ) -> None:
        self.name = name
        self.description = description
        self.modality = modality
        self.cost = cost
        self.handler = handler

    def invoke(self, arguments: dict[str, Any]) -> ToolResult:
        """Ejecuta una herramienta simulada y devuelve un resultado trazable."""
        try:
            content = self.handler(arguments)
            return ToolResult(content=content, cost=self.cost, success=True)
        except Exception as error:
            return ToolResult(
                content=f"Error controlado: {error}",
                cost=self.cost,
                success=False,
            )

In [ ]:
def search_text(arguments: dict[str, Any]) -> str:
    """Recupera evidencia textual desde un índice local simulado."""
    topic = str(arguments.get("topic", "")).lower()
    knowledge = {
        "clima": "El informe textual indica lluvia intensa.",
        "reunion": "La agenda textual indica una reunión a las 10.",
        "alarma": "El manual indica revisar primero la señal acústica.",
    }
    return knowledge.get(topic, "No se encontró evidencia textual relevante.")


def inspect_image(arguments: dict[str, Any]) -> str:
    """Inspecciona una referencia visual simulada."""
    image_id = str(arguments.get("image_id", ""))
    images = {
        "img_lluvia": "La imagen muestra calles mojadas y paraguas.",
        "img_sala": "La imagen muestra una sala vacía.",
        "img_tablero": "La imagen muestra una luz roja en el tablero.",
    }
    return images.get(image_id, "La referencia visual no contiene evidencia útil.")


def inspect_audio(arguments: dict[str, Any]) -> str:
    """Inspecciona una referencia acústica simulada."""
    audio_id = str(arguments.get("audio_id", ""))
    audio = {
        "aud_lluvia": "El audio contiene lluvia y truenos.",
        "aud_alarma": "El audio contiene una alarma intermitente.",
        "aud_silencio": "El audio no contiene eventos destacados.",
    }
    return audio.get(audio_id, "La referencia acústica no contiene evidencia útil.")

In [ ]:
TOOLS = {
    "search_text": Tool(
        name="search_text",
        description="Busca evidencia textual local",
        modality="text",
        cost=0.5,
        handler=search_text,
    ),
    "inspect_image": Tool(
        name="inspect_image",
        description="Inspecciona una referencia visual",
        modality="image",
        cost=1.0,
        handler=inspect_image,
    ),
    "inspect_audio": Tool(
        name="inspect_audio",
        description="Inspecciona una referencia acústica",
        modality="audio",
        cost=1.0,
        handler=inspect_audio,
    ),
}

print("Herramientas disponibles:", list(TOOLS))

#### **Tareas controladas**

Cada tarea define herramientas mínimas y una palabra clave esperada.

El evaluador observa acciones y respuesta final. No depende de razonamientos internos.

In [ ]:
TASKS = [
    {
        "task_id": "t1",
        "query": "Determina si está lloviendo usando la evidencia disponible.",
        "references": {
            "topic": "clima",
            "image_id": "img_lluvia",
            "audio_id": "aud_lluvia",
        },
        "required_tools": {"inspect_image", "inspect_audio"},
        "expected_keyword": "lluvia",
    },
    {
        "task_id": "t2",
        "query": "Determina si existe una alarma activa.",
        "references": {
            "topic": "alarma",
            "image_id": "img_tablero",
            "audio_id": "aud_alarma",
        },
        "required_tools": {"inspect_audio"},
        "expected_keyword": "alarma",
    },
    {
        "task_id": "t3",
        "query": "Determina la hora de la reunión.",
        "references": {
            "topic": "reunion",
            "image_id": "img_sala",
            "audio_id": "aud_silencio",
        },
        "required_tools": {"search_text"},
        "expected_keyword": "10",
    },
]

#### **Cadena fija**

La cadena fija ejecuta todas las herramientas en un orden constante.

Su ventaja es la simplicidad. Su costo aparece cuando usa herramientas innecesarias.

In [ ]:
def run_fixed_chain(
    task: dict[str, Any],
    tools: dict[str, Tool],
) -> AgentState:
    """Ejecuta todas las herramientas sin adaptar la secuencia."""
    state = AgentState(query=task["query"], status="ejecutando")

    for tool_name in ["search_text", "inspect_image", "inspect_audio"]:
        tool = tools[tool_name]
        result = tool.invoke(task["references"])
        state.current_step += 1
        state.remaining_budget -= result.cost
        state.actions.append(
            ActionRecord(
                step=state.current_step,
                tool_name=tool_name,
                arguments=task["references"],
                result=result.content,
                cost=result.cost,
                success=result.success,
            )
        )

    state.final_answer = " ".join(action.result for action in state.actions)
    state.finished = True
    state.status = "finalizado"
    return state

#### **Workflow con reglas**

El workflow selecciona una ruta predefinida mediante palabras clave.

Es eficiente cuando la tarea coincide con una regla conocida.

In [ ]:
def select_workflow_tools(query: str) -> list[str]:
    """Selecciona una ruta predefinida mediante reglas explícitas."""
    normalized = query.lower()

    if "lloviendo" in normalized:
        return ["inspect_image", "inspect_audio"]

    if "alarma" in normalized:
        return ["inspect_audio"]

    if "reunión" in normalized:
        return ["search_text"]

    return ["search_text"]


def run_workflow(
    task: dict[str, Any],
    tools: dict[str, Tool],
) -> AgentState:
    """Ejecuta una ruta seleccionada por reglas."""
    state = AgentState(query=task["query"], status="ejecutando")

    for tool_name in select_workflow_tools(task["query"]):
        tool = tools[tool_name]
        result = tool.invoke(task["references"])
        state.current_step += 1
        state.remaining_budget -= result.cost
        state.actions.append(
            ActionRecord(
                step=state.current_step,
                tool_name=tool_name,
                arguments=task["references"],
                result=result.content,
                cost=result.cost,
                success=result.success,
            )
        )

    state.final_answer = " ".join(action.result for action in state.actions)
    state.finished = True
    state.status = "finalizado"
    return state

#### **Agente dinámico**

El agente selecciona herramientas usando el objetivo, la trayectoria y el presupuesto.

El planificador incluye ruido controlado para estudiar robustez.

In [ ]:
def plan_action(
    state: AgentState,
    task: dict[str, Any],
    planner_noise: float,
    rng: random.Random,
) -> str | None:
    """Selecciona la siguiente herramienta o decide terminar."""
    used_tools = {action.tool_name for action in state.actions}
    required_tools = set(task["required_tools"])

    if required_tools.issubset(used_tools):
        return None

    remaining_tools = sorted(required_tools - used_tools)

    if rng.random() < planner_noise:
        candidates = sorted(set(TOOLS) - used_tools)
        if candidates:
            return rng.choice(candidates)

    if remaining_tools:
        return remaining_tools[0]

    return None


def run_agent(
    task: dict[str, Any],
    tools: dict[str, Tool],
    max_steps: int = 4,
    planner_noise: float = 0.0,
    seed: int = 228,
) -> AgentState:
    """Ejecuta un agente con planificación y terminación explícita."""
    rng = random.Random(seed)
    logger = TraceLogger()
    state = AgentState(
        query=task["query"],
        max_steps=max_steps,
        status="ejecutando",
    )

    logger.add(
        "inicio",
        {
            "consulta": state.query,
            "presupuesto": state.remaining_budget,
        },
    )

    while not state.finished:
        if state.current_step >= state.max_steps:
            state.status = "presupuesto_de_pasos_agotado"
            state.finished = True
            break

        tool_name = plan_action(
            state=state,
            task=task,
            planner_noise=planner_noise,
            rng=rng,
        )

        if tool_name is None:
            state.status = "evidencia_suficiente"
            state.finished = True
            break

        tool = tools[tool_name]

        if state.remaining_budget < tool.cost:
            state.status = "presupuesto_de_costo_agotado"
            state.finished = True
            break

        result = tool.invoke(task["references"])
        state.current_step += 1
        state.remaining_budget -= result.cost

        action = ActionRecord(
            step=state.current_step,
            tool_name=tool_name,
            arguments=task["references"],
            result=result.content,
            cost=result.cost,
            success=result.success,
        )
        state.actions.append(action)
        state.observations.append(
            Observation(
                modality=tool.modality,
                content=result.content,
                source=tool.name,
                confidence=1.0 if result.success else 0.0,
            )
        )

        logger.add(
            "accion",
            {
                "paso": state.current_step,
                "herramienta": tool_name,
                "resultado": result.content,
                "costo": result.cost,
            },
        )

    state.final_answer = " ".join(
        observation.content
        for observation in state.observations
    )

    logger.add(
        "fin",
        {
            "estado": state.status,
            "respuesta": state.final_answer,
        },
    )

    trace_path = RESULTS_DIR / f"trace_{task['task_id']}_{seed}.jsonl"
    logger.save(trace_path)
    return state

#### **Métricas**

Se evalúan éxito, costo, pasos y llamadas innecesarias.

Una llamada es innecesaria cuando usa una herramienta fuera del conjunto mínimo de la tarea.

In [ ]:
def evaluate_run(
    system_name: str,
    task: dict[str, Any],
    state: AgentState,
) -> dict[str, Any]:
    """Evalúa utilidad y eficiencia de una ejecución."""
    final_answer = state.final_answer or ""
    used_tools = [action.tool_name for action in state.actions]
    required_tools = set(task["required_tools"])

    success = task["expected_keyword"].lower() in final_answer.lower()
    unnecessary_calls = sum(
        tool_name not in required_tools
        for tool_name in used_tools
    )
    total_cost = sum(action.cost for action in state.actions)

    return {
        "sistema": system_name,
        "tarea": task["task_id"],
        "exito": int(success),
        "pasos": state.current_step,
        "llamadas": len(used_tools),
        "llamadas_innecesarias": unnecessary_calls,
        "costo": total_cost,
        "estado": state.status,
    }

#### **Experimento comparativo**

Se ejecutan los tres sistemas sobre las mismas tareas.

El agente se evalúa también con ruido para medir sensibilidad del planificador.

In [ ]:
records: list[dict[str, Any]] = []

for task in TASKS:
    fixed_state = run_fixed_chain(task, TOOLS)
    workflow_state = run_workflow(task, TOOLS)
    agent_state = run_agent(
        task,
        TOOLS,
        max_steps=4,
        planner_noise=0.0,
        seed=SEED,
    )
    noisy_state = run_agent(
        task,
        TOOLS,
        max_steps=4,
        planner_noise=0.35,
        seed=SEED + int(task["task_id"][-1]),
    )

    records.append(evaluate_run("cadena_fija", task, fixed_state))
    records.append(evaluate_run("workflow", task, workflow_state))
    records.append(evaluate_run("agente", task, agent_state))
    records.append(evaluate_run("agente_con_ruido", task, noisy_state))

for record in records:
    print(record)

In [ ]:
def summarize_records(
    records: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Resume resultados por sistema."""
    systems = sorted({record["sistema"] for record in records})
    summary: list[dict[str, Any]] = []

    for system in systems:
        selected = [
            record
            for record in records
            if record["sistema"] == system
        ]
        summary.append(
            {
                "sistema": system,
                "tasa_de_exito": sum(item["exito"] for item in selected) / len(selected),
                "pasos_medios": sum(item["pasos"] for item in selected) / len(selected),
                "costo_medio": sum(item["costo"] for item in selected) / len(selected),
                "llamadas_innecesarias_medias": sum(
                    item["llamadas_innecesarias"]
                    for item in selected
                ) / len(selected),
            }
        )

    return summary


summary = summarize_records(records)

for item in summary:
    print(item)

#### **Ablación del presupuesto**

El límite de pasos controla autonomía, costo y riesgo de bucles.

Se compara el agente con dos, tres y cuatro pasos.

In [ ]:
budget_records: list[dict[str, Any]] = []

for max_steps in [2, 3, 4]:
    for task in TASKS:
        state = run_agent(
            task,
            TOOLS,
            max_steps=max_steps,
            planner_noise=0.35,
            seed=SEED + max_steps + int(task["task_id"][-1]),
        )
        record = evaluate_run(
            f"agente_presupuesto_{max_steps}",
            task,
            state,
        )
        record["presupuesto_de_pasos"] = max_steps
        budget_records.append(record)

for record in budget_records:
    print(record)

#### **Lectura académica de los resultados**

- La cadena fija prioriza cobertura y simplicidad.
- El workflow prioriza control y eficiencia dentro de rutas conocidas.
- El agente prioriza adaptación y puede reducir llamadas cuando el planificador es correcto.
El ruido muestra que autonomía sin presupuesto y terminación explícita puede aumentar costo y errores.

#### **Amenazas a la validez**

- Las herramientas son simuladas y no representan errores perceptuales reales.
- El planificador usa reglas y ruido controlado. No representa toda la variabilidad de un LLM.
- Las tareas son pequeñas y el conjunto mínimo de herramientas es conocido.

El cuaderno estudia arquitectura y causalidad experimental. No estima desempeño en producción.

#### **Preguntas de desarrollo**

1. ¿Qué propiedad convierte a un workflow en un agente?

2. ¿Por qué una mayor autonomía puede reducir reproducibilidad?

3. ¿Cómo distinguir una llamada útil de una llamada innecesaria?

4. ¿Qué ocurre si el agente termina con evidencia parcial?

5. ¿Cómo cambia la evaluación cuando una herramienta tiene efectos irreversibles?

6. ¿Qué información mínima debe conservar una traza para permitir auditoría?.

#### **Exportación reproducible**

Las métricas y trazas se guardan para permitir auditoría y repetición del experimento.

In [ ]:
def write_csv(
    path: Path,
    rows: list[dict[str, Any]],
) -> None:
    """Guarda una lista de diccionarios en formato CSV."""
    if not rows:
        return

    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


write_csv(RESULTS_DIR / "resultados.csv", records)
write_csv(RESULTS_DIR / "resumen.csv", summary)
write_csv(RESULTS_DIR / "ablacion_presupuesto.csv", budget_records)

metadata = {
    "curso": "MCC225",
    "semana": 13,
    "cuaderno": "Cuaderno28-MCC225",
    "tema": "Arquitectura reproducible de un agente multimodal",
    "semilla": SEED,
    "modo": "CPU sin APIs externas",
    "numero_de_tareas": len(TASKS),
}

with (RESULTS_DIR / "metadata.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print("Resultados exportados en:", RESULTS_DIR)

#### **Conclusión**

- Una cadena fija ofrece simplicidad.
- Un workflow ofrece control estructural.
- Un agente ofrece adaptación dinámica.
- La autonomía solo se justifica cuando mejora utilidad sin perder trazabilidad, presupuesto y control de terminación.